In [323]:
from pathlib import Path
from typing import List, Any
from langchain_community.document_loaders import Docx2txtLoader
from langchain_community.document_loaders.excel import UnstructuredExcelLoader
from langchain_community.document_loaders import PyPDFLoader, TextLoader, CSVLoader


class IngestionAgent:
    def load_all_docs(self,data_dir: str) -> List[Any]:
        """
        Load all supported files from the data directory 
        Convert to LangChain document structure.
        Supported: PDF, TXT, CSV, Excel, Word.
        """
        # Root data folder
        data_path = Path(data_dir).resolve()
        print(f"Status: Data path: {data_path}")
        
        #defined empty document list
        documents = []

        # PDF files
        pdf_files = list(data_path.glob('**/*.pdf'))
        print(f"Status: Found {len(pdf_files)} PDF files: {[str(f) for f in pdf_files]}")
        for pdf_file in pdf_files:
            print(f"Status: Loading PDF: {pdf_file}")
            try:
                loader = PyPDFLoader(str(pdf_file))
                loaded = loader.load()
                print(f"Status: Loaded {len(loaded)} PDF docs from {pdf_file}")
                documents.extend(loaded)
            except Exception as e:
                print(f"[ERROR] Failed to load PDF {pdf_file}: {e}")

        # Word files
        docx_files = list(data_path.glob('**/*.docx'))
        print(f"Status: Found {len(docx_files)} Word files: {[str(f) for f in docx_files]}")
        for docx_file in docx_files:
            print(f"Status: Loading Word: {docx_file}")
            try:
                loader = Docx2txtLoader(str(docx_file))
                loaded = loader.load()
                print(f"Status: Loaded {len(loaded)} Word docs from {docx_file}")
                documents.extend(loaded)
            except Exception as e:
                print(f"[ERROR] Failed to load Word {docx_file}: {e}")

        # CSV files
        csv_files = list(data_path.glob('**/*.csv'))
        print(f"Status: Found {len(csv_files)} CSV files: {[str(f) for f in csv_files]}")
        for csv_file in csv_files:
            print(f"Status: Loading CSV: {csv_file}")
            try:
                loader = CSVLoader(str(csv_file))
                loaded = loader.load()
                print(f"Status: Loaded {len(loaded)} CSV docs from {csv_file}")
                documents.extend(loaded)
            except Exception as e:
                print(f"[ERROR] Failed to load CSV {csv_file}: {e}")

        # Excel files
        xlsx_files = list(data_path.glob('**/*.xlsx'))
        print(f"Status: Found {len(xlsx_files)} Excel files: {[str(f) for f in xlsx_files]}")
        for xlsx_file in xlsx_files:
            print(f"Status: Loading Excel: {xlsx_file}")
            try:
                loader = UnstructuredExcelLoader(str(xlsx_file))
                loaded = loader.load()
                print(f"Status: Loaded {len(loaded)} Excel docs from {xlsx_file}")
                documents.extend(loaded)
            except Exception as e:
                print(f"[ERROR] Failed to load Excel {xlsx_file}: {e}")

        # TXT files
        txt_files = list(data_path.glob('**/*.txt'))
        print(f"Status: Found {len(txt_files)} TXT files: {[str(f) for f in txt_files]}")
        for txt_file in txt_files:
            print(f"Status: Loading TXT: {txt_file}")
            try:
                loader = TextLoader(str(txt_file))
                loaded = loader.load()
                print(f"Status: Loaded {len(loaded)} TXT docs from {txt_file}")
                documents.extend(loaded)
            except Exception as e:
                print(f"[ERROR] Failed to load TXT {txt_file}: {e}")

        
        print(f"Status: Total loaded documents: {len(documents)}")
        return documents

# Example usage
if __name__ == "__main__":
    ingest=IngestionAgent()
    docs = ingest.load_all_docs("./data")
    print(f"Loaded {len(docs)} documents.")
    print("Example document:", docs[0] if docs else None)

Ignoring wrong pointing object 6 0 (offset 0)
Ignoring wrong pointing object 150 0 (offset 0)
Ignoring wrong pointing object 151 0 (offset 0)


Status: Data path: /Users/pavankumarb/Documents/My Learning/DocENTmcp/data
Status: Found 2 PDF files: ['/Users/pavankumarb/Documents/My Learning/DocENTmcp/data/pdf/Group 4 – Data Protection and Privacy.pdf', '/Users/pavankumarb/Documents/My Learning/DocENTmcp/data/pdf/attention all you need.pdf']
Status: Loading PDF: /Users/pavankumarb/Documents/My Learning/DocENTmcp/data/pdf/Group 4 – Data Protection and Privacy.pdf
Status: Loaded 18 PDF docs from /Users/pavankumarb/Documents/My Learning/DocENTmcp/data/pdf/Group 4 – Data Protection and Privacy.pdf
Status: Loading PDF: /Users/pavankumarb/Documents/My Learning/DocENTmcp/data/pdf/attention all you need.pdf
Status: Loaded 15 PDF docs from /Users/pavankumarb/Documents/My Learning/DocENTmcp/data/pdf/attention all you need.pdf
Status: Found 1 Word files: ['/Users/pavankumarb/Documents/My Learning/DocENTmcp/data/docx/Project Proposal Smart Travel-Adventure Planner Bot.docx']
Status: Loading Word: /Users/pavankumarb/Documents/My Learning/DocEN

In [320]:
from typing import List, Any
from langchain_text_splitters import RecursiveCharacterTextSplitter
from sentence_transformers import SentenceTransformer
import numpy as np
#from agents.ingestion_agent import load_all_docs

class Embedding:
    def __init__(self, model_name: str = "all-MiniLM-L6-v2", chunk_size: int = 1000, chunk_overlap: int = 200):
        self.chunk_size = chunk_size
        self.chunk_overlap = chunk_overlap
        self.model = SentenceTransformer(model_name)
        print(f"Status: Loaded embedding model: {model_name}")

    def chunk_documents(self, documents: List[Any]) -> List[Any]:
        splitter = RecursiveCharacterTextSplitter(
            chunk_size=self.chunk_size,
            chunk_overlap=self.chunk_overlap,
            length_function=len,
            separators=["\n\n", "\n", " ", ""]
        )
        chunks = splitter.split_documents(documents)
        print(f"Status: Split {len(documents)} documents into {len(chunks)} chunks.")
        return chunks

    def embed_chunks(self, chunks: List[Any]) -> np.ndarray:
        texts = [chunk.page_content for chunk in chunks]
        print(f"Status: Generating embeddings for {len(texts)} chunks...")
        embeddings = self.model.encode(texts, show_progress_bar=True)
        print(f"Status: Embeddings shape: {embeddings.shape}")
        return embeddings

# Example usage
if __name__ == "__main__":
    emb_pipe = Embedding()
    chunks = emb_pipe.chunk_documents(docs)
    embeddings = emb_pipe.embed_chunks(chunks)
    print("Status: Example embedding:", chunks[0] if len(chunks) > 0 else None ,embeddings[0] if len(embeddings) > 0 else None)

Status: Loaded embedding model: all-MiniLM-L6-v2
Status: Split 9835 documents into 13058 chunks.
Status: Generating embeddings for 13058 chunks...


Batches: 100%|██████████| 409/409 [00:20<00:00, 19.92it/s]


Status: Embeddings shape: (13058, 384)
Status: Example embedding: page_content='Data 
Protection 
and Privacy
Group 4
1.Pavan Kumar Boddupally 
2.Kusuma Kankanala 
3.Kevin Rodriguez 
4. Md Istihad Alam 
5. Benjamin Castro' metadata={'producer': 'macOS Version 15.6.1 (Build 24G90) Quartz PDFContext', 'creator': 'Keynote', 'creationdate': "D:20251122123256Z00'00'", 'title': 'Group 4 – Data Protection and Privacy.pdf', 'author': 'Pavan Kumar Boddupally', 'moddate': "D:20251122123256Z00'00'", 'source': '/Users/pavankumarb/Documents/My Learning/DocENTmcp/data/pdf/Group 4 – Data Protection and Privacy.pdf', 'total_pages': 18, 'page': 0, 'page_label': '1'} [-4.30258326e-02  3.00627183e-02 -3.32171805e-02 -5.40830940e-02
  3.85903120e-02  5.35517447e-02  7.38651529e-02 -4.47132513e-02
  2.43362207e-02  1.60339754e-02  1.03239611e-01 -7.51614664e-03
  2.98262481e-02 -7.35344961e-02 -1.27182361e-02  4.44146283e-02
 -2.32653487e-02  1.46100251e-02 -3.63918580e-02 -7.91264251e-02
 -6.64726272e-02 

In [ ]:
import os
import faiss
import numpy as np
import pickle
from typing import List, Any
from sentence_transformers import SentenceTransformer
#from src.embedding import EmbeddingPipeline

class FaissVectorStore:
    def __init__(self, persist_dir: str = "faiss_store", embedding_model: str = "all-MiniLM-L6-v2", chunk_size: int = 1000, chunk_overlap: int = 200):
        self.persist_dir = persist_dir
        os.makedirs(self.persist_dir, exist_ok=True)
        self.index = None
        self.metadata = []
        self.embedding_model = embedding_model
        self.model = SentenceTransformer(embedding_model)
        self.chunk_size = chunk_size
        self.chunk_overlap = chunk_overlap
        print(f"[INFO] Loaded embedding model: {embedding_model}")

    def build_from_documents(self, documents: List[Any]):
        print(f"[INFO] Building vector store from {len(documents)} raw documents...")
        emb_pipe = EmbeddingPipeline(model_name=self.embedding_model, chunk_size=self.chunk_size, chunk_overlap=self.chunk_overlap)
        chunks = emb_pipe.chunk_documents(documents)
        embeddings = emb_pipe.embed_chunks(chunks)
        metadatas = [{"text": chunk.page_content} for chunk in chunks]
        self.add_embeddings(np.array(embeddings).astype('float32'), metadatas)
        self.save()
        print(f"[INFO] Vector store built and saved to {self.persist_dir}")

    def add_embeddings(self, embeddings: np.ndarray, metadatas: List[Any] = None):
        dim = embeddings.shape[1]
        if self.index is None:
            self.index = faiss.IndexFlatL2(dim)
        self.index.add(embeddings)
        if metadatas:
            self.metadata.extend(metadatas)
        print(f"[INFO] Added {embeddings.shape[0]} vectors to Faiss index.")

    def save(self):
        faiss_path = os.path.join(self.persist_dir, "faiss.index")
        meta_path = os.path.join(self.persist_dir, "metadata.pkl")
        faiss.write_index(self.index, faiss_path)
        with open(meta_path, "wb") as f:
            pickle.dump(self.metadata, f)
        print(f"[INFO] Saved Faiss index and metadata to {self.persist_dir}")

    def load(self):
        faiss_path = os.path.join(self.persist_dir, "faiss.index")
        meta_path = os.path.join(self.persist_dir, "metadata.pkl")
        self.index = faiss.read_index(faiss_path)
        with open(meta_path, "rb") as f:
            self.metadata = pickle.load(f)
        print(f"[INFO] Loaded Faiss index and metadata from {self.persist_dir}")

    def search(self, query_embedding: np.ndarray, top_k: int = 5):
        D, I = self.index.search(query_embedding, top_k)
        results = []
        for idx, dist in zip(I[0], D[0]):
            meta = self.metadata[idx] if idx < len(self.metadata) else None
            results.append({"index": idx, "distance": dist, "metadata": meta})
        return results

    def query(self, query_text: str, top_k: int = 5):
        print(f"[INFO] Querying vector store for: '{query_text}'")
        query_emb = self.model.encode([query_text]).astype('float32')
        return self.search(query_emb, top_k=top_k)

# Example usage
if __name__ == "__main__":
    store = FaissVectorStore("faiss_store")
    store.build_from_documents(docs)
    store.load()
    print(store.query("What is attention mechanism?", top_k=3))
    print()

[INFO] Loaded embedding model: all-MiniLM-L6-v2
[INFO] Building vector store from 9835 raw documents...
Status: Loaded embedding model: all-MiniLM-L6-v2
Status: Split 9835 documents into 13058 chunks.
Status: Generating embeddings for 13058 chunks...


Batches: 100%|██████████| 409/409 [00:22<00:00, 18.44it/s]


Status: Embeddings shape: (13058, 384)
[INFO] Added 13058 vectors to Faiss index.
[INFO] Saved Faiss index and metadata to faiss_store
[INFO] Vector store built and saved to faiss_store
[INFO] Loaded Faiss index and metadata from faiss_store
[INFO] Querying vector store for: 'What is attention mechanism?'
[{'index': np.int64(29), 'distance': np.float32(0.7285826), 'metadata': {'text': '3.2 Attention\nAn attention function can be described as mapping a query and a set of key-value pairs to an output,\nwhere the query, keys, values, and output are all vectors. The output is computed as a weighted sum\n3'}}, {'index': np.int64(66), 'distance': np.float32(0.8640241), 'metadata': {'text': 'Attention Visualizations\nInput-Input Layer5\nIt\nis\nin\nthis\nspirit\nthat\na\nmajority\nof\nAmerican\ngovernments\nhave\npassed\nnew\nlaws\nsince\n2009\nmaking\nthe\nregistration\nor\nvoting\nprocess\nmore\ndifficult\n.\n<EOS>\n<pad>\n<pad>\n<pad>\n<pad>\n<pad>\n<pad>\nIt\nis\nin\nthis\nspirit\nthat\na

In [333]:
import os
from dotenv import load_dotenv
#from src.vectorstore import FaissVectorStore
#from langchain_groq import ChatGroq
from langchain_ollama import ChatOllama

class RAGSearch:
    def __init__(self, persist_dir: str = "faiss_store", embedding_model: str = "all-MiniLM-L6-v2", llm_model: str = "gemma2-9b-it"):
        self.vectorstore = FaissVectorStore(persist_dir, embedding_model)
        # Load or build vectorstore
        faiss_path = os.path.join(persist_dir, "faiss.index")
        meta_path = os.path.join(persist_dir, "metadata.pkl")
        if not (os.path.exists(faiss_path) and os.path.exists(meta_path)):
            #from data_loader import load_all_documents
            ingest=IngestionAgent()
            docs = ingest.load_all_docs("data")
            self.vectorstore.build_from_documents(docs)
        else:
            self.vectorstore.load()
        #groq_api_key = ""
        self.llm = ChatOllama(model="llama3.1")
        print(f"Status Ollama LLM initialized: {self.llm}")

    def search_and_summarize(self, query: str, top_k: int = 5) -> str:
        results = self.vectorstore.query(query, top_k=top_k)
        texts = [r["metadata"].get("text", "") for r in results if r["metadata"]]
        context = "\n\n".join(texts)
        if not context:
            return "No relevant documents found."
        prompt = f"""Summarize the following context for the query: '{query}'\n\nContext:\n{context}\n\nSummary:"""
        response = self.llm.invoke([prompt])
        return response.content

# Example usage
if __name__ == "__main__":
    

    rag_search = RAGSearch()
    query = "Explain about model context protocol in 5 points ?"
    summary = rag_search.search_and_summarize(query, top_k=3)
    print("Summary:", summary)
    

[INFO] Loaded embedding model: all-MiniLM-L6-v2
[INFO] Loaded Faiss index and metadata from faiss_store
Status Ollama LLM initialized: model='llama3.1'
[INFO] Querying vector store for: 'Explain about model context protocol in 5 points ?'
Summary: Here is a summary of the Model Context Protocol in 5 points:

1. **Key Components**: The Model Context Protocol consists of several key components, including Base Protocol, Lifecycle Management, Authorization, Server Features, Client Features, and Utilities.
2. **Implementation Requirements**: All implementations MUST support the base protocol and lifecycle management components, while other components MAY be implemented based on specific application needs.
3. **Modular Design**: The modular design allows for clear separation of concerns and enables rich interactions between clients and servers, supporting exactly the features needed by each implementation.
4. **Reserved Key Names**: Certain key names are reserved by MCP for protocol-level me

In [17]:


import uuid
from typing import Annotated, Sequence, TypedDict, Optional, Any
from enum import Enum

from langchain_core.messages import BaseMessage, HumanMessage, AIMessage
from langchain_ollama import ChatOllama
from langgraph.graph import StateGraph, START, END
from langgraph.graph.message import add_messages
from langgraph.checkpoint.memory import InMemorySaver
from langgraph.types import interrupt, Command
from langgraph.store.memory import InMemoryStore

# Import your existing agents
try:
    from src.summary_agent import SummaryAgent
    from src.qa_agent import QAAgent
    from src.extraction_agent import ExtractionAgent
    from src.star_agent import StarAgent
    from src.report_agent import ReportAgent
except ImportError:
    import sys
    from pathlib import Path
    project_root = Path(__file__).parent.parent
    if str(project_root) not in sys.path:
        sys.path.insert(0, str(project_root))
    from src.summary_agent import SummaryAgent
    from src.qa_agent import QAAgent
    from src.report_agent import ReportAgent
    from src.extraction_agent import ExtractionAgent
    from src.star_agent import StarAgent


In [12]:
class SubAgentType(str, Enum):
    QA = "qa"
    SUMMARY = "summary"
    STAR = "star"
    EXTRACTION = "extraction"
    REPORT = "report"


class DeepAgentState(TypedDict):
    """Shared state across all agents and subagents."""
    messages: Annotated[Sequence[BaseMessage], add_messages]
    documents: str
    user_query: str
    current_subagent: Optional[SubAgentType]
    subagent_results: dict
    approval_status: dict
    final_report: str
    analysis_complete: bool

In [13]:
class SubAgentRunner:
    """Runs individual agents and returns results."""
    
    def __init__(self):
        self.qa = QAAgent()
        self.summary = SummaryAgent()
        self.star = StarAgent()
        self.extraction = ExtractionAgent()
        self.report = ReportAgent()
    
    def run_qa(self, query: str) -> str:
        """Run QA agent with vector search."""
        return self.qa.search_and_summarize(query, top_k=5)
    
    def run_summary(self, text: str) -> str:
        """Run Summary agent."""
        return self.summary.summarize(text)
    
    def run_star(self, text: str) -> str:
        """Run STAR analysis."""
        return self.star.generate_star(text)
    
    def run_extraction(self, text: str) -> str:
        """Run data extraction."""
        return self.extraction.extract_tables(text)
    
    def run_report(self, texts: list) -> str:
        """Generate final report from all results."""
        return self.report.generate_report(texts)


In [38]:
class DeepAgent:
    def __init__(self, model_name: str = "llama3.1"):
        self.llm = ChatOllama(model=model_name)
        self.runner = SubAgentRunner()
        self.checkpointer = InMemorySaver()
        self.store = InMemoryStore()

        self.graph = self._build_graph()
        self.compiled_graph = self.graph.compile(
            checkpointer=self.checkpointer,
            store=self.store
        )
        def analyze_query(self, user_query: str) -> list:
            query_lower = user_query.lower()
            required_agents = []

            qa_keywords = [
                "what", "who", "where", "when", "why", "how",
                "find", "search", "look for", "question", "ask",
                "specific", "detail", "information about", "tell me",
                "?", "answer", "explain", "define", "describe"
            ]

            summary_keywords = [
                "summary", "summarize", "overview", "brief", "short",
                "key points", "main", "highlights", "outline", "condensed",
                "gist", "essence", "recap", "digest"
            ]

            star_keywords = [
                "star", "situation", "task", "action", "result",
                "achieved", "accomplished", "success", "events",
                "decision", "problem", "solve", "overcome", "challenge",
                "situation and task"
            ]

            extraction_keywords = [
                "extract", "table", "data", "structured", "numbers",
                "metrics", "statistics", "values", "figures", "numeric",
                "list", "enumerate", "items", "columns", "rows"
            ]

            report_keywords = [
                "report", "analysis", "comprehensive", "detailed", "full",
                "complete", "overall", "everything", "all", "total",
                "combine", "merge", "aggregate", "holistic"
            ]

            if any(keyword in query_lower for keyword in qa_keywords):
                required_agents.append(SubAgentType.QA)

            if any(keyword in query_lower for keyword in summary_keywords):
                required_agents.append(SubAgentType.SUMMARY)

            if any(keyword in query_lower for keyword in star_keywords):
                required_agents.append(SubAgentType.STAR)

            if any(keyword in query_lower for keyword in extraction_keywords):
                required_agents.append(SubAgentType.EXTRACTION)

            if any(keyword in query_lower for keyword in report_keywords):
                required_agents.append(SubAgentType.REPORT)

            if not required_agents:
                required_agents = [SubAgentType.QA, SubAgentType.SUMMARY]

            return required_agents

    # -------------------
    # BUILD GRAPH
    # -------------------
    def _build_graph(self) -> StateGraph:
        workflow = StateGraph(DeepAgentState)

        # --- Node: parse input ---
        def parse_input(state: DeepAgentState) -> dict:
            latest_msg = state["messages"][-1]
            content = latest_msg.content if isinstance(latest_msg, HumanMessage) else ""

            if "DOCUMENT:" in content and "QUERY:" in content:
                doc_part, query_part = content.split("QUERY:")
                documents = doc_part.replace("DOCUMENT:", "").strip()
                query = query_part.strip()
            else:
                documents = content
                query = content[:100]

            msg = AIMessage(content=f"📄 Parsing input: {len(documents)} chars...")
            return {
                "documents": documents,
                "user_query": query,
                "messages": state["messages"] + [msg],
                "subagent_results": {},
                "approval_status": {}
            }

        workflow.add_node("parse_input", parse_input)

        # --- Node generator for subagents ---
        def make_subagent_node(subagent_name: SubAgentType, runner_func):
            def node(state: DeepAgentState) -> dict:
                if not state["documents"]:
                    return {"subagent_results": {**state["subagent_results"], subagent_name.value: None}}

                result = runner_func(state["documents"] if subagent_name != SubAgentType.QA else state["user_query"])

                # Trigger human approval
                try:
                    interrupt({
                        "type": "subagent_result",
                        "subagent": subagent_name.value,
                        "result": result[:300],
                        "full_result": result,
                        "options": ["approve", "reject", "edit"]
                    })
                except Exception as e:
                    # The interrupt exception will be caught by compiled_graph
                    raise

                # Normally unreachable until resume_with_approval
                return {
                    "current_subagent": subagent_name,
                    "subagent_results": {**state["subagent_results"], subagent_name.value: result},
                    "approval_status": {**state["approval_status"], subagent_name.value: "approved"},
                    "messages": state["messages"] + [AIMessage(content=f"✅ {subagent_name.value} done")]
                }

            return node

        # Add QA, Summary, STAR, Extraction nodes
        workflow.add_node("qa_subagent", make_subagent_node(SubAgentType.QA, self.runner.run_qa))
        workflow.add_node("summary_subagent", make_subagent_node(SubAgentType.SUMMARY, self.runner.run_summary))
        workflow.add_node("star_subagent", make_subagent_node(SubAgentType.STAR, self.runner.run_star))
        workflow.add_node("extraction_subagent", make_subagent_node(SubAgentType.EXTRACTION, self.runner.run_extraction))

        # --- Node: report ---
        def report_node(state: DeepAgentState) -> dict:
            approved_texts = [r for r in state["subagent_results"].values() if r]
            final_report = self.runner.run_report(approved_texts) if approved_texts else "No approved results."

            try:
                interrupt({
                    "type": "final_report",
                    "subagent": "report",
                    "result": final_report[:300],
                    "full_result": final_report,
                    "options": ["approve", "reject"]
                })
            except Exception as e:
                raise

            return {
                "current_subagent": SubAgentType.REPORT,
                "final_report": final_report,
                "analysis_complete": True,
                "approval_status": {**state["approval_status"], "report": "approved"},
                "messages": state["messages"] + [AIMessage(content="📋 Report done")]
            }

        workflow.add_node("report_subagent", report_node)

        # --- EDGES ---
        workflow.add_edge(START, "parse_input")
        workflow.add_edge("parse_input", "qa_subagent")
        workflow.add_edge("qa_subagent", "summary_subagent")
        workflow.add_edge("summary_subagent", "star_subagent")
        workflow.add_edge("star_subagent", "extraction_subagent")
        workflow.add_edge("extraction_subagent", "report_subagent")
        workflow.add_edge("report_subagent", END)

        return workflow

    # -------------------
    # INVOKE / RESUME
    # -------------------
    def invoke(self, documents: str, query: str, thread_id: str = None) -> dict:
        if thread_id is None:
            thread_id = str(uuid.uuid4())

        user_input = f"DOCUMENT:\n{documents}\n\nQUERY:\n{query}"
        initial_state = {
            "messages": [HumanMessage(content=user_input)],
            "documents": "",
            "user_query": "",
            "current_subagent": None,
            "subagent_results": {},
            "approval_status": {},
            "final_report": "",
            "analysis_complete": False
        }

        config = {"configurable": {"thread_id": thread_id}}

        try:
            return self.compiled_graph.invoke(initial_state, config)
        except Exception as e:
            # Usually an interrupt for human approval
            print(">>> Human approval required:", e)
            raise

    def resume_with_approval(self, thread_id: str, decision: str, edited_value: str = None) -> dict:
        config = {"configurable": {"thread_id": thread_id}}
        resume_value = {"decision": decision, "edited_value": edited_value}
        return self.compiled_graph.invoke(Command(resume=resume_value), config)

    def get_thread_state(self, thread_id: str) -> dict:
        config = {"configurable": {"thread_id": thread_id}}
        return self.compiled_graph.get_state(config)


# =======================
# USAGE EXAMPLE
# =======================
if __name__ == "__main__":
    # Initialize the deep agent
    agent = DeepAgent()

    # Input document and query
    documents = "This is a test document about AI and ML."
    query = "Summarize the document and extract STAR points."

    # Start the workflow
    thread_id = str(uuid.uuid4())
    try:
        state = agent.invoke(documents, query, thread_id=thread_id)
    except Exception as e:
        # Interrupt captured, workflow paused for approval
        state = e.args[0] if e.args else {}
    
    # Loop until analysis is complete
    while not state.get("analysis_complete", False):
        # Check for pending interrupt
        interrupt_info = state.get("__interrupt__")
        if interrupt_info:
            intr_data = interrupt_info[0].value  # get the first interrupt
            subagent = intr_data.get("subagent")
            print(f">>> Auto-approving {subagent} step.")
            
            # Auto-approve; could replace with "edit" and edited_value if needed
            decision = "approve"
            edited_value = None

            # Resume workflow with approval
            state = agent.resume_with_approval(thread_id, decision, edited_value)
        else:
            break  # No interrupt, workflow finished

    # Final report
    print("\n===== FINAL REPORT =====")
    print(state.get("final_report"))


[INFO] Loaded embedding model: all-MiniLM-L6-v2
[INFO] Loaded Faiss index and metadata from faiss_store
Status Ollama LLM initialized: model='llama3.1'
Status: Ollama LLM initialized for SummaryAgent: llama3.1
Status: Ollama LLM initialized for SummaryAgent: llama3.1
[INFO] Ollama LLM initialized for ExtractionAgent: llama3.1
[INFO] Querying vector store for: 'Summarize the document and extract STAR points.'
>>> Auto-approving qa step.
[INFO] Querying vector store for: 'Summarize the document and extract STAR points.'
>>> Auto-approving summary step.
>>> Auto-approving star step.
>>> Auto-approving extraction step.
>>> Auto-approving report step.

===== FINAL REPORT =====
It appears that this is a chatbot's response to a user prompt to generate a report based on provided documents. The chatbot has analyzed the content, identified key insights and findings, and extracted STAR points, tables, numeric data, and structured information.

Here are some observations from the output:

1. **Key

In [33]:
import uuid
from typing import Annotated, Sequence, TypedDict, Optional
from enum import Enum
from langchain_core.messages import BaseMessage, HumanMessage, AIMessage
from langchain_ollama import ChatOllama
from langgraph.graph import StateGraph, START, END
from langgraph.graph.message import add_messages
from langgraph.checkpoint.memory import InMemorySaver
from langgraph.store.memory import InMemoryStore

# Import your existing agents
try:
    from src.summary_agent import SummaryAgent
    from src.qa_agent import QAAgent
    from src.extraction_agent import ExtractionAgent
    from src.star_agent import StarAgent
    from src.report_agent import ReportAgent
except ImportError:
    import sys
    from pathlib import Path
    project_root = Path(__file__).parent.parent
    if str(project_root) not in sys.path:
        sys.path.insert(0, str(project_root))
    from src.summary_agent import SummaryAgent
    from src.qa_agent import QAAgent
    from src.report_agent import ReportAgent
    from src.extraction_agent import ExtractionAgent
    from src.star_agent import StarAgent


class SubAgentType(str, Enum):
    QA = "qa"
    SUMMARY = "summary"
    STAR = "star"
    EXTRACTION = "extraction"
    REPORT = "report"


class DeepAgentState(TypedDict):
    """Shared state across all agents and subagents."""
    messages: Annotated[Sequence[BaseMessage], add_messages]
    documents: str  # Retrieved from FAISS
    user_query: str
    current_subagent: Optional[SubAgentType]
    subagent_results: dict
    approval_status: dict
    final_report: str
    analysis_complete: bool
    required_agents: list


class SubAgentRunner:
    """Runs individual agents and returns results."""
    
    def __init__(self):
        self.qa = QAAgent()
        self.summary = SummaryAgent()
        self.star = StarAgent()
        self.extraction = ExtractionAgent()
        self.report = ReportAgent()
    
    def run_qa(self, query: str) -> str:
        """Run QA agent with vector search."""
        return self.qa.search_and_summarize(query, top_k=5)
    
    def run_summary(self, text: str) -> str:
        """Run Summary agent."""
        return self.summary.summarize(text)
    
    def run_star(self, text: str) -> str:
        """Run STAR analysis."""
        return self.star.generate_star(text)
    
    def run_extraction(self, text: str) -> str:
        """Run data extraction."""
        return self.extraction.extract_tables(text)
    
    def run_report(self, texts: list) -> str:
        """Generate final report from all results."""
        return self.report.generate_report(texts)


class DeepAgent:
    """
    Intelligent DeepAgent that routes to specific subagents based on user query.
    Documents are automatically retrieved from FAISS vector store.
    
    Usage:
        agent = DeepAgent()
        agent.invoke("Your query here")  # Streams results as agents complete
    """
    
    def __init__(self, model_name: str = "llama3.1"):
        self.model_name = model_name
        self.llm = ChatOllama(model=model_name)
        self.runner = SubAgentRunner()
        self.checkpointer = InMemorySaver()
        self.store = InMemoryStore()

        self.graph = self._build_graph()
        self.compiled_graph = self.graph.compile(
            checkpointer=self.checkpointer,
            store=self.store
        )
    
    def analyze_query(self, user_query: str) -> list:
        """
        Analyze user query and determine which subagents to run.
        
        Args:
            user_query (str): The user's query
        
        Returns:
            List of SubAgentType enums to execute
        """
        query_lower = user_query.lower()
        required_agents = []
        
        # QA Keywords - Direct question answering
        qa_keywords = [
            "what", "who", "where", "when", "why", "how",
            "find", "search", "look for", "question", "ask",
            "specific", "detail", "information about", "tell me",
            "?", "answer", "explain", "define", "describe"
        ]
        
        # SUMMARY Keywords - Overview/summary requests
        summary_keywords = [
            "summary", "summarize", "overview", "brief", "short",
            "key points", "main", "highlights", "outline", "condensed",
            "gist", "essence", "recap", "digest"
        ]
        
        # STAR Keywords - Business/event analysis
        star_keywords = [
            "star", "situation", "task", "action", "result",
            "achieved", "accomplished", "success", "events",
            "decision", "problem", "solve", "overcome", "challenge",
            "situation and task"
        ]
        
        # EXTRACTION Keywords - Data/table extraction
        extraction_keywords = [
            "extract", "table", "data", "structured", "numbers",
            "metrics", "statistics", "values", "figures", "numeric",
            "list", "enumerate", "items", "columns", "rows"
        ]
        
        # REPORT Keywords - Comprehensive analysis
        report_keywords = [
            "report", "analysis", "comprehensive", "detailed", "full",
            "complete", "overall", "everything", "all", "total",
            "combine", "merge", "aggregate", "holistic"
        ]
        
        # Check for each agent type
        if any(keyword in query_lower for keyword in qa_keywords):
            required_agents.append(SubAgentType.QA)
        
        if any(keyword in query_lower for keyword in summary_keywords):
            required_agents.append(SubAgentType.SUMMARY)
        
        if any(keyword in query_lower for keyword in star_keywords):
            required_agents.append(SubAgentType.STAR)
        
        if any(keyword in query_lower for keyword in extraction_keywords):
            required_agents.append(SubAgentType.EXTRACTION)
        
        if any(keyword in query_lower for keyword in report_keywords):
            required_agents.append(SubAgentType.REPORT)
        
        # If no agents matched, default to QA + SUMMARY
        if not required_agents:
            required_agents = [SubAgentType.QA, SubAgentType.SUMMARY]
        
        return required_agents
    
    def _build_graph(self) -> StateGraph:
        workflow = StateGraph(DeepAgentState)

        # --- Node: Retrieve documents from FAISS and analyze query ---
        def retrieve_and_analyze(state: DeepAgentState) -> dict:
            user_query = state["user_query"]
            
            # Retrieve documents from FAISS using QA agent
            print(f"📚 Retrieving documents from FAISS for query: {user_query[:50]}...")
            retrieved_docs = self.runner.qa.vectorstore.query(user_query, top_k=10)
            
            # Extract text from retrieved documents
            documents = "\n\n".join([
                doc.get("metadata", {}).get("text", "")
                for doc in retrieved_docs if doc.get("metadata")
            ])
            
            if not documents:
                documents = "No relevant documents found in FAISS."
            
            # Analyze query to determine required agents
            required_agents = self.analyze_query(user_query)
            agent_names = ", ".join([a.value for a in required_agents])
            
            msg = AIMessage(
                content=f"📚 Retrieved {len(retrieved_docs)} documents from FAISS ({len(documents)} chars)\n"
                f"🤖 Running agents: {agent_names}"
            )
            
            return {
                "documents": documents,
                "messages": state["messages"] + [msg],
                "subagent_results": {},
                "approval_status": {},
                "required_agents": required_agents
            }

        workflow.add_node("retrieve_and_analyze", retrieve_and_analyze)

        # --- Node generator for subagents ---
        def make_subagent_node(subagent_name: SubAgentType, runner_func):
            def node(state: DeepAgentState) -> dict:
                # Skip if this agent is not required
                if subagent_name not in state.get("required_agents", []):
                    print(f"⏭️  Skipping {subagent_name.value} (not required)")
                    return {
                        "subagent_results": {**state["subagent_results"], subagent_name.value: None},
                        "approval_status": {**state["approval_status"], subagent_name.value: "skipped"}
                    }
                
                if not state["documents"]:
                    return {
                        "subagent_results": {**state["subagent_results"], subagent_name.value: None},
                        "approval_status": {**state["approval_status"], subagent_name.value: "skipped"}
                    }

                print(f"🤖 Running {subagent_name.value}...")
                
                # Run the appropriate agent
                if subagent_name == SubAgentType.QA:
                    result = runner_func(state["user_query"])
                else:
                    result = runner_func(state["documents"])

                return {
                    "current_subagent": subagent_name,
                    "subagent_results": {**state["subagent_results"], subagent_name.value: result},
                    "approval_status": {**state["approval_status"], subagent_name.value: "completed"},
                    "messages": state["messages"] + [AIMessage(content=f"✅ {subagent_name.value} completed")]
                }

            return node

        # Add QA, Summary, STAR, Extraction nodes
        workflow.add_node("qa_subagent", make_subagent_node(SubAgentType.QA, self.runner.run_qa))
        workflow.add_node("summary_subagent", make_subagent_node(SubAgentType.SUMMARY, self.runner.run_summary))
        workflow.add_node("star_subagent", make_subagent_node(SubAgentType.STAR, self.runner.run_star))
        workflow.add_node("extraction_subagent", make_subagent_node(SubAgentType.EXTRACTION, self.runner.run_extraction))

        # --- Node: report ---
        def report_node(state: DeepAgentState) -> dict:
            # Skip if report not required
            if SubAgentType.REPORT not in state.get("required_agents", []):
                print(f"⏭️  Skipping report (not required)")
                return {
                    "final_report": "",
                    "analysis_complete": True,
                    "approval_status": {**state["approval_status"], "report": "skipped"}
                }
            
            print("📋 Generating report...")
            
            approved_texts = [r for r in state["subagent_results"].values() if r]
            final_report = self.runner.run_report(approved_texts) if approved_texts else "No results to generate report."

            return {
                "current_subagent": SubAgentType.REPORT,
                "final_report": final_report,
                "analysis_complete": True,
                "approval_status": {**state["approval_status"], "report": "completed"},
                "messages": state["messages"] + [AIMessage(content="📋 Report generated")]
            }

        workflow.add_node("report_subagent", report_node)

        # --- EDGES ---
        workflow.add_edge(START, "retrieve_and_analyze")
        workflow.add_edge("retrieve_and_analyze", "qa_subagent")
        workflow.add_edge("qa_subagent", "summary_subagent")
        workflow.add_edge("summary_subagent", "star_subagent")
        workflow.add_edge("star_subagent", "extraction_subagent")
        workflow.add_edge("extraction_subagent", "report_subagent")
        workflow.add_edge("report_subagent", END)

        return workflow
    
    def invoke(self, user_query: str, thread_id: str = None) -> None:
        """
        Stream deep agent analysis with real-time output.
        Results are printed as each agent completes (not all at the end).
        
        Args:
            user_query (str): The user's query/question
            thread_id (str): Optional thread ID for persistence
        """
        if thread_id is None:
            thread_id = str(uuid.uuid4())

        initial_state = {
            "messages": [HumanMessage(content=f"Query: {user_query}")],
            "documents": "",
            "user_query": user_query,
            "current_subagent": None,
            "subagent_results": {},
            "approval_status": {},
            "final_report": "",
            "analysis_complete": False,
            "required_agents": []
        }

        config = {"configurable": {"thread_id": thread_id}}

        try:
            # Stream mode="updates" shows only changed fields after each node
            for chunk in self.compiled_graph.stream(initial_state, config, stream_mode="updates"):
                # chunk is now just a dict with updates, not a tuple
                # Check if chunk is a dictionary
                if isinstance(chunk, dict):
                    # Show agent completions as they happen
                    if "approval_status" in chunk:
                        for agent_name, status in chunk["approval_status"].items():
                            if status == "completed":
                                result = chunk.get("subagent_results", {}).get(agent_name)
                                if result:
                                    print(f"\n✅ {agent_name.upper()} COMPLETED:")
                                    print(f"   {result[:150]}...")  # Show first 150 chars
                            elif status == "skipped":
                                print(f"⏭️  {agent_name.upper()} SKIPPED")
                    
                    # Show final report
                    if "final_report" in chunk and chunk["final_report"]:
                        print(f"\n📋 FINAL REPORT:")
                        print(f"   {chunk['final_report'][:200]}...")
            
            print("\n✅ Analysis complete!")
            
        except Exception as e:
            print("❌ Error:", str(e))
            raise
    def get_thread_state(self, thread_id: str) -> dict:
        """Get the current state of a thread."""
        config = {"configurable": {"thread_id": thread_id}}
        return self.compiled_graph.get_state(config)


if __name__ == "__main__":
    agent = DeepAgent()
    
    # Example 1: Direct Question (runs QA only)
    print("=" * 80)
    print("EXAMPLE 1: Direct Question")
    print("=" * 80)
    agent.invoke("What was the revenue in Q3?")

[INFO] Loaded embedding model: all-MiniLM-L6-v2
[INFO] Loaded Faiss index and metadata from faiss_store
Status Ollama LLM initialized: model='llama3.1'
Status: Ollama LLM initialized for SummaryAgent: llama3.1
Status: Ollama LLM initialized for SummaryAgent: llama3.1
[INFO] Ollama LLM initialized for ExtractionAgent: llama3.1
EXAMPLE 1: Direct Question
📚 Retrieving documents from FAISS for query: What was the revenue in Q3?...
[INFO] Querying vector store for: 'What was the revenue in Q3?'
🤖 Running qa...
[INFO] Querying vector store for: 'What was the revenue in Q3?'
⏭️  Skipping summary (not required)
⏭️  Skipping star (not required)
⏭️  Skipping extraction (not required)
⏭️  Skipping report (not required)

✅ Analysis complete!


# temparay


In [28]:
if __name__ == "__main__":
    agent = DeepAgent()
    # Example 2: Summary Request (runs SUMMARY)
    print("=" * 80)
    print("EXAMPLE 2: Summary Request")
    print("=" * 80)
    result2 = agent.invoke("Give me a brief summary of the report")
    print(f"✅ Required Agents: {[a.value for a in result2.get('required_agents', [])]}")
    print(f"📊 Completed Agents: {result2['approval_status']}\n")
    
    # Example 3: STAR Analysis (runs STAR)
    print("=" * 80)
    print("EXAMPLE 3: STAR Analysis")
    print("=" * 80)
    result3 = agent.invoke("What were the key achievements and decisions made?")
    print(f"✅ Required Agents: {[a.value for a in result3.get('required_agents', [])]}")
    print(f"📊 Completed Agents: {result3['approval_status']}\n")

[INFO] Loaded embedding model: all-MiniLM-L6-v2
[INFO] Loaded Faiss index and metadata from faiss_store
Status Ollama LLM initialized: model='llama3.1'
Status: Ollama LLM initialized for SummaryAgent: llama3.1
Status: Ollama LLM initialized for SummaryAgent: llama3.1
[INFO] Ollama LLM initialized for ExtractionAgent: llama3.1
EXAMPLE 2: Summary Request
📚 Retrieving documents from FAISS for query: Give me a brief summary of the report...
[INFO] Querying vector store for: 'Give me a brief summary of the report'
⏭️  Skipping qa (not required)
🤖 Running summary...
⏭️  Skipping star (not required)
⏭️  Skipping extraction (not required)
📋 Generating report...

✅ Analysis complete!
✅ Required Agents: ['summary', 'report']
📊 Completed Agents: {'qa': 'skipped', 'summary': 'completed', 'star': 'skipped', 'extraction': 'skipped', 'report': 'completed'}

EXAMPLE 3: STAR Analysis
📚 Retrieving documents from FAISS for query: What were the key achievements and decisions made?...
[INFO] Querying vecto

In [26]:
if __name__ == "__main__":
    agent = DeepAgent()
    
    documents = """
    Q3 2024 Financial Report
    
    Revenue: $50M (↑25% YoY)
    Operating Expenses: $20M (↓10% YoY)
    Net Profit: $11M (↑22% margin)
    
    Achievements:
    - Launched 3 new products
    - Expanded to 5 new markets
    - Hired 50 team members
    
    Risks:
    - Market competition
    - Supply chain issues
    - Currency fluctuations
    """
  # Example 2: Summary Request (runs SUMMARY)
    print("=" * 60)
    print("EXAMPLE 2: Summary Request")
    print("=" * 60)
    result2 = agent.invoke(documents, "Give me a brief summary of the report")
    print(f"✅ Required Agents: {[a.value for a in result2.get('required_agents', [])]}")
    print(f"📊 Results: {result2['approval_status']}\n")

[INFO] Loaded embedding model: all-MiniLM-L6-v2
[INFO] Loaded Faiss index and metadata from faiss_store
Status Ollama LLM initialized: model='llama3.1'
Status: Ollama LLM initialized for SummaryAgent: llama3.1
Status: Ollama LLM initialized for SummaryAgent: llama3.1
[INFO] Ollama LLM initialized for ExtractionAgent: llama3.1
EXAMPLE 2: Summary Request
⏭️ Skipping qa (not required)
🤖 Running summary...
⏭️ Skipping star (not required)
⏭️ Skipping extraction (not required)
📋 Generating report...
✅ Required Agents: ['summary', 'report']
📊 Results: {'qa': 'skipped', 'summary': 'completed', 'star': 'skipped', 'extraction': 'skipped', 'report': 'completed'}



1+1